In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg
from dotenv import load_dotenv
from pgvector.psycopg import register_vector
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sqlalchemy import create_engine, text

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

DSN = (
    f"postgresql://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}"
    f"@{os.getenv('POSTGRES_HOST', 'localhost')}:{os.getenv('POSTGRES_PORT', '5432')}/{os.environ['POSTGRES_DB']}"
)
engine = create_engine(DSN.replace("postgresql://", "postgresql+psycopg://"))

MODEL = "BAAI/bge-small-en-v1.5"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
with psycopg.connect(DSN, autocommit=True) as conn:
    for statement in (ROOT / "db/init/02_embeddings.sql").read_text().split(";"):
        if statement.strip():
            conn.execute(statement)

In [ ]:
products = pd.read_sql("SELECT product_id, product_name, aisle, department FROM raw.v_products ORDER BY product_id", engine)
products["text"] = products["product_name"] + ". Aisle: " + products["aisle"] + ". Department: " + products["department"] + "."
print(products.shape)
products[["product_id", "text"]].head()

In [ ]:
model = SentenceTransformer(MODEL)
emb = model.encode(products["text"].tolist(), batch_size=128, normalize_embeddings=True, show_progress_bar=True).astype(np.float32)
print(emb.shape)

(ROOT / "models").mkdir(exist_ok=True)
np.save(ROOT / "models/product_embeddings_bge_small.npy", emb)

In [ ]:
with psycopg.connect(DSN) as conn:
    register_vector(conn)
    conn.execute("TRUNCATE app.product_embeddings")
    with conn.cursor() as cur:
        cur.executemany(
            "INSERT INTO app.product_embeddings (product_id, model, embedding) VALUES (%s, %s, %s)",
            [(int(p), MODEL, e) for p, e in zip(products["product_id"], emb)],
        )
    conn.execute("CREATE INDEX IF NOT EXISTS product_embeddings_hnsw ON app.product_embeddings USING hnsw (embedding vector_cosine_ops)")
    conn.execute("ANALYZE app.product_embeddings")

pd.read_sql("SELECT count(*) AS rows, max(model) AS model, max(vector_dims(embedding)) AS dims FROM app.product_embeddings", engine)

In [ ]:
SIMILAR_SQL = text("""
SELECT p.product_name, p.aisle, p.department,
       1 - (e.embedding <=> (SELECT embedding FROM app.product_embeddings WHERE product_id = :pid)) AS similarity
FROM app.product_embeddings e
JOIN raw.v_products p USING (product_id)
WHERE e.product_id <> :pid
ORDER BY e.embedding <=> (SELECT embedding FROM app.product_embeddings WHERE product_id = :pid)
LIMIT :k
""")


def similar(product_id, k=10):
    return pd.read_sql(SIMILAR_SQL, engine, params={"pid": int(product_id), "k": k})


for keyword in ["Bag of Organic Bananas", "Organic Whole Milk", "Chocolate Chip Cookies", "Sparkling Water", "Dog Food"]:
    match = products[products["product_name"].str.contains(keyword, case=False)].head(1)
    if match.empty:
        continue
    row = match.iloc[0]
    print(f"\n== {row['product_name']}  [{row['aisle']} / {row['department']}]")
    display(similar(row["product_id"]))

In [ ]:
name_emb = model.encode(products["product_name"].tolist(), batch_size=256, normalize_embeddings=True, show_progress_bar=True).astype(np.float32)

rng = np.random.default_rng(0)
sample = rng.choice(len(products), 3000, replace=False)
sims = name_emb[sample] @ name_emb.T
sims[np.arange(len(sample)), sample] = -1
top = np.argsort(-sims, axis=1)[:, :10]

aisle = products["aisle"].to_numpy()
department = products["department"].to_numpy()

rows = []
for k in (1, 5, 10):
    nb = top[:, :k]
    rows.append({"k": k,
                 "same_aisle": (aisle[nb] == aisle[sample][:, None]).mean(),
                 "same_department": (department[nb] == department[sample][:, None]).mean()})
check = pd.DataFrame(rows).set_index("k")
check.loc["random"] = [(products["aisle"].value_counts(normalize=True) ** 2).sum(),
                       (products["department"].value_counts(normalize=True) ** 2).sum()]
check.round(3)

In [ ]:
check.drop("random").plot.bar(rot=0, figsize=(8, 3.8), color=["#4c72b0", "#55a868"], title="Nearest neighbours by product name only: share in the same aisle / department")
plt.axhline(check.loc["random", "same_aisle"], ls="--", c="#4c72b0", alpha=.6)
plt.axhline(check.loc["random", "same_department"], ls="--", c="#55a868", alpha=.6)
plt.xlabel("k (neighbours)")

In [ ]:
idx = rng.choice(len(products), 6000, replace=False)
xy = TSNE(n_components=2, init="pca", perplexity=30, random_state=0).fit_transform(PCA(50, random_state=0).fit_transform(emb[idx]))
deps = department[idx]

fig, ax = plt.subplots(figsize=(12, 8))
for i, d in enumerate(sorted(set(deps))):
    m = deps == d
    ax.scatter(xy[m, 0], xy[m, 1], s=6, color=plt.cm.tab20(i % 20), marker="o" if i < 20 else "^", label=d)
ax.legend(markerscale=3, bbox_to_anchor=(1.01, 1), loc="upper left")
ax.set_title("Product embeddings (t-SNE, 6000 random products), coloured by department")
ax.grid(False)

In [ ]:
from src.recommender.knn import ItemKNN

train = pd.read_csv(ROOT / "data/processed/train.csv")
knn = ItemKNN().fit(train, history_users=[])

row_of = {int(p): i for i, p in enumerate(products["product_id"])}
cand_rows = np.array([row_of[int(p)] for p in knn.items])
cand_emb = emb[cand_rows]
cand_dep = department[cand_rows]

pick = rng.choice(len(cand_rows), 1000, replace=False)
text_sims = cand_emb[pick] @ cand_emb.T
text_sims[np.arange(len(pick)), pick] = -1
text_nb = np.argsort(-text_sims, axis=1)[:, :10]

co_nb = np.array([[n for n in knn.neighbor_idx[i] if n != i][:10] for i in pick])

overlap = np.mean([len(set(a) & set(b)) / 10 for a, b in zip(text_nb, co_nb)])
result = pd.DataFrame({
    "text neighbours": {"same department": (cand_dep[text_nb] == cand_dep[pick][:, None]).mean()},
    "co-purchase neighbours": {"same department": (cand_dep[co_nb] == cand_dep[pick][:, None]).mean()},
})
print(f"Overlap@10 between text and co-purchase neighbours: {overlap:.3f}")
result.round(3)